Cosmos3-Edge inference: navigation instructions + predicted future frame\n\n**Hardware note:** Cosmos3-Edge is only officially supported on Linux with an NVIDIA GPU (Ampere/Hopper/Blackwell, BF16). It will not run on this Mac (no CUDA) — the cells below are correct for a CUDA+Linux machine (e.g. a cloud GPU box), but you'll need to move `./cosmos3-edge` and this notebook there to actually execute them.\n\nThe model exposes two separate paths, both used below:\n- **Reasoning (text)** — served via vLLM's OpenAI-compatible API (`--omni`), used here to turn the image + prompt into step-by-step navigation instructions.\n- **Image-to-video (future prediction)** — the `diffusers` `Cosmos3OmniPipeline`, used here to generate a short clip of what the robot's camera would see next; we take the last frame as the predicted future frame.

In [ ]:
%pip install -q -U \
    "diffusers @ git+https://github.com/huggingface/diffusers.git" \
    "vllm[omni]" \
    accelerate \
    av \
    cosmos_guardrail \
    huggingface_hub \
    imageio \
    imageio-ffmpeg \
    openai \
    pillow \
    requests \
    torch \
    torchvision \
    transformers

In [ ]:
from pathlib import Path

# Local Cosmos3-Edge weights (downloaded in cosmos3import.ipynb)
local_model_path = "./cosmos3-edge"

# Inference input: the image the robot currently sees, and what to ask about it
image_path = "/Users/raymondshaw/Documents/VLA Research/tests/IMG_8525.jpg"
prompt = "Analyze this environment. Give me step-by-step instructions to navigate to the blue trashcan."

vllm_host = "0.0.0.0"
vllm_port = 8000
vllm_base_url = f"http://localhost:{vllm_port}/v1"

In [ ]:
import subprocess
import time

import requests

# Serve the reasoning (autoregressive) tower over an OpenAI-compatible API.
# Runs in the background for the life of this kernel; terminated in the last cell.
vllm_process = subprocess.Popen([
    "vllm", "serve", local_model_path,
    "--omni",
    "--host", vllm_host,
    "--port", str(vllm_port),
    "--init-timeout", "1800",
])


def wait_for_vllm(timeout=1800, interval=5):
    start = time.time()
    while time.time() - start < timeout:
        if vllm_process.poll() is not None:
            raise RuntimeError(f"vLLM server exited early with code {vllm_process.returncode}")
        try:
            if requests.get(f"{vllm_base_url}/models", timeout=5).status_code == 200:
                return
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(interval)
    raise TimeoutError("vLLM server did not become ready in time")


wait_for_vllm()
print("vLLM server ready")

In [ ]:
import base64

import openai
from PIL import Image

image = Image.open(image_path).convert("RGB")
image_bytes = Path(image_path).read_bytes()
image_data_url = "data:image/jpeg;base64," + base64.b64encode(image_bytes).decode()

client = openai.OpenAI(api_key="EMPTY", base_url=vllm_base_url)
reasoning_model_id = client.models.list().data[0].id

reasoning_prompt = (
    f"{prompt} Respond with a numbered, step-by-step list of specific navigation "
    "instructions (direction, distance/landmarks, and any obstacles to avoid) "
    "to reach the target from the robot's current position."
)

response = client.chat.completions.create(
    model=reasoning_model_id,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_data_url}},
                {"type": "text", "text": reasoning_prompt},
            ],
        },
    ],
    max_tokens=4096,
)

navigation_instructions = response.choices[0].message.content
print("Cosmos3-Edge navigation instructions:")
print(navigation_instructions)

In [ ]:
import json

import torch
from diffusers import Cosmos3OmniPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler

# Reuse the model's standard negative prompt if it was downloaded alongside the weights.
negative_prompt_path = Path(local_model_path) / "assets" / "negative_prompt.json"
if negative_prompt_path.exists():
    negative_prompt = json.dumps(json.loads(negative_prompt_path.read_text()))
else:
    negative_prompt = "low quality, distorted, flickering, physically inconsistent, blurry"

# Ground the video generation in the same target + plan used for the reasoning call above.
video_prompt = (
    f"{prompt} Continue the scene from the robot's current camera viewpoint as it follows "
    f"this plan: {navigation_instructions}"
)

pipe = Cosmos3OmniPipeline.from_pretrained(
    local_model_path,
    torch_dtype=torch.bfloat16,
    enable_safety_checker=True,
)
pipe.to("cuda")
pipe.scheduler = UniPCMultistepScheduler.from_config(
    pipe.scheduler.config, flow_shift=12.0, use_karras_sigmas=False
)

result = pipe(
    prompt=video_prompt,
    negative_prompt=negative_prompt,
    image=image,
    num_frames=49,
    height=480,
    width=832,
    fps=24.0,
    num_inference_steps=20,
    guidance_scale=6.0,
    generator=torch.Generator(device="cuda").manual_seed(0),
)

In [ ]:
import numpy as np
from diffusers.utils import export_to_video

video_frames = result.video
# Unwrap a batch-of-one if the pipeline returned frames nested per-sample.
if len(video_frames) == 1 and isinstance(video_frames[0], (list, tuple)):
    video_frames = video_frames[0]

export_to_video(video_frames, "predicted_future.mp4", fps=24, macro_block_size=1)

predicted_future_frame = video_frames[-1]
if isinstance(predicted_future_frame, np.ndarray):
    predicted_future_frame = Image.fromarray(predicted_future_frame)

predicted_future_frame.save("predicted_future_frame.png")
predicted_future_frame

In [ ]:
# Run once you're done with the kernel to stop the background vLLM server.
vllm_process.terminate()
vllm_process.wait(timeout=30)